In [ ]:
# Fix the loop feature. For each file, re-derive loop_num and
# bin_within_loop by anchoring to the ORIGINAL (pre-dropna) row count,
# correcting the boundary shift caused by leading-NaN trimming.
# Files with interior NaN rows (broken time-adjacency) are excluded.
# Only the two loop columns are changed; all other columns are preserved.
# Overwrites topomap_data/{train,val}/labels.csv in place.
import pandas as pd
import numpy as np
from pathlib import Path

def recompute_loops(split):
    csv_dir = Path(split)                       # original CSVs (train/ or val/)
    labels_path = Path('topomap_data') / split / 'labels.csv'
    labels = pd.read_csv(labels_path)

    new_rows = []
    excluded_files = []

    for fname, group in labels.groupby('filename'):
        # fname is like "P001_T01"; reconstruct the original CSV name
        pid, tid = fname.split('_')
        csv_file = csv_dir / f'Features_{pid}-{tid}.csv'

        d = pd.read_csv(csv_file)
        nan_mask = d.isna().any(axis=1).values
        nan_idx = np.where(nan_mask)[0]

        # ---- Exclude files with INTERIOR NaNs (the 5 problem files) ----
        is_leading = list(nan_idx) == list(range(len(nan_idx)))
        if len(nan_idx) > 0 and not is_leading:
            excluded_files.append(fname)
            continue

        n_dropped = len(nan_idx)               # how many leading rows dropna removed
        original_total = len(d)                # TRUE recording length (pre-drop)

        # ---- True loop boundaries from ORIGINAL length, 4 equal loops ----
        # account for the same end-remainder trim the pipeline applied
        surviving_total = original_total - n_dropped
        remainder = surviving_total % 4
        usable_original = original_total - remainder  # mirror pipeline's trim logic
        bins_per_loop = usable_original // 4

        group = group.sort_values('row_idx').reset_index(drop=True)

        for _, r in group.iterrows():
            post_drop_idx = r['row_idx']             # position in the .npy
            true_idx = post_drop_idx + n_dropped     # position in ORIGINAL recording
            # assign loop from true position
            loop_num = min(true_idx // bins_per_loop, 3) + 1
            bin_within = true_idx % bins_per_loop

            rr = r.copy()
            rr['loop_num'] = loop_num
            rr['bin_within_loop'] = bin_within
            new_rows.append(rr)

    new_labels = pd.DataFrame(new_rows)
    new_labels.to_csv(labels_path, index=False)
    print(f"{split}: rewrote {len(new_labels)} rows, "
          f"excluded {len(excluded_files)} interior-NaN files: {excluded_files}")
    return new_labels

train_new = recompute_loops('train')
val_new = recompute_loops('val')

train: rewrote 244620 rows, excluded 4 interior-NaN files: ['P030_T09', 'P038_T14', 'P045_T19', 'P056_T24']
val: rewrote 83400 rows, excluded 1 interior-NaN files: ['P091_T14']


In [ ]:
# Sanity-check the rewrite: compare old vs new row counts (should drop
# only by the excluded files' rows), count how many loop_num values
# changed (should be non-zero, proving the fix took effect), and show
# the new loop distribution across loops 1-4.
old = pd.read_csv('topomap_data/train/labels_backup.csv')
new = pd.read_csv('topomap_data/train/labels.csv')
print("Old rows:", len(old), "New rows:", len(new))
print("Rows where loop_num changed:",
      (old.set_index(['filename','row_idx'])['loop_num']
         .ne(new.set_index(['filename','row_idx'])['loop_num'])).sum())
print("\nNew loop distribution:")
print(new['loop_num'].value_counts().sort_index())

Old rows: 245100 New rows: 244620
Rows where loop_num changed: 6879

New loop distribution:
loop_num
1    59022
2    61155
3    61155
4    63288
Name: count, dtype: int64


In [ ]:
# Spot-check a few files: print the loop_num sequence per file to
# confirm clean ascending blocks (1s, then 2s, 3s, 4s) with no
# interleaving. Confirms loop boundaries are assigned correctly.
import pandas as pd
new = pd.read_csv('topomap_data/train/labels.csv')

# For a few files, look at the loop_num sequence to confirm it's 1,1,1...2,2,2...3...4
for fname in new['filename'].unique()[:3]:
    g = new[new['filename'] == fname].sort_values('row_idx')
    print(fname, "loop sequence:", g['loop_num'].values)

P001_T01 loop sequence: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4]
P001_T02 loop sequence: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4]
P001_T03 loop sequence: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]


In [4]:
old = pd.read_csv('topomap_data/train/labels_backup.csv')
new = pd.read_csv('topomap_data/train/labels.csv')
merged = old.merge(new, on=['filename', 'row_idx'], suffixes=('_old', '_new'))
for col in ['arousal', 'heartrate', 'gsr', 'treatment_id', 'participant_id']:
    changed = (merged[f'{col}_old'] != merged[f'{col}_new']).sum()
    print(f"{col}: {changed} rows changed (should be 0)")

arousal: 0 rows changed (should be 0)
heartrate: 0 rows changed (should be 0)
gsr: 387 rows changed (should be 0)
treatment_id: 0 rows changed (should be 0)
participant_id: 0 rows changed (should be 0)


In [ ]:
# Confirm only the loop columns changed. Compare arousal, heartrate,
# gsr, treatment_id, participant_id between backup and new labels.
# All should report 0 changes (note: gsr may show tiny sub-1e-21
# float round-trip noise, which is not a real change).
merged = old.merge(new, on=['filename', 'row_idx'], suffixes=('_old', '_new'))
diff = merged[merged['gsr_old'] != merged['gsr_new']]
print(f"{len(diff)} rows flagged")
print("\nMax absolute difference:", (diff['gsr_old'] - diff['gsr_new']).abs().max())
print("\nSample of the differences:")
print(diff[['filename', 'row_idx', 'gsr_old', 'gsr_new']].head(10).to_string())

387 rows flagged

Max absolute difference: 4.235164736271502e-22

Sample of the differences:
      filename  row_idx       gsr_old       gsr_new
1790  P005_T12       10 -2.059500e-07 -2.059500e-07
1819  P005_T12       39  3.140000e-09  3.140000e-09
1820  P005_T12       40  1.660000e-09  1.660000e-09
1822  P005_T12       42 -5.340000e-09 -5.340000e-09
1833  P005_T12       53  5.947000e-08  5.947000e-08
1839  P005_T12       59  5.567000e-08  5.567000e-08
1842  P005_T12       62  4.749000e-08  4.749000e-08
1843  P005_T12       63  5.199000e-08  5.199000e-08
1866  P005_T12       86  4.832000e-08  4.832000e-08
1887  P005_T12      107  4.003000e-08  4.003000e-08
